# Reddit Kaggle Competition: Adrian Barrios

Multi-label emotion classification over 28 categories. Metric: **Macro ROC AUC**.

### Architecture
Fine-tuned **DistilBERT** (`distilbert-base-uncased`) with a linear classification head over the 28 emotion labels. `problem_type="multi_label_classification"` gives us `BCEWithLogitsLoss` directly from `transformers`.

### Training
- AdamW (`lr=2e-5`, weight decay 0.01 on non-bias/LayerNorm params).
- Linear warmup (10% of steps) + linear decay.
- Gradient clipping at 1.0, mixed-precision on GPU.
- The full training run is executed on a Modal T4 via `modal_train_bert.py` (3 epochs). This notebook reproduces the architecture locally for quick verification.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

DATA_DIR = '../data/'
PRETRAINED = 'distilbert-base-uncased'
TOKENIZER_OUT = 'tokenizer'
BATCH_SIZE = 16   # local sanity batch; Modal run uses 32
MAX_LEN = 128
EPOCHS = 1        # local sanity epochs; Modal run uses 3
LR = 2e-5
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'Using device: {DEVICE}')

## 1. Load the Data

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_kaggle.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_kaggle.csv'))

emotion_cols = [c for c in train_df.columns if c not in ['ID', 'text']]
num_labels = len(emotion_cols)

print(f'Training samples: {len(train_df)}')
print(f'Test samples: {len(test_df)}')
print(f'Total emotions: {num_labels}')

## 2. Tokenizer
We use the WordPiece tokenizer that ships with `distilbert-base-uncased`. Saving it to `tokenizer/` makes the inference notebook fully offline-reproducible.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)
tokenizer.save_pretrained(TOKENIZER_OUT)
print(f'Tokenizer saved to {TOKENIZER_OUT}/')

## 3. Dataset and DataLoader
Pre-tokenize once into `input_ids` + `attention_mask` tensors so the training loop is just GPU compute.

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, df, is_test=False):
        enc = tokenizer(
            df['text'].astype(str).tolist(),
            padding='max_length', truncation=True, max_length=MAX_LEN,
            return_tensors='pt',
        )
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.is_test = is_test
        if not is_test:
            self.labels = torch.tensor(df[emotion_cols].values.astype(np.float32))

    def __len__(self):
        return self.input_ids.size(0)

    def __getitem__(self, idx):
        item = {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
        }
        if not self.is_test:
            item['labels'] = self.labels[idx]
        return item

train_split, val_split = train_test_split(train_df, test_size=0.1, random_state=SEED)
train_loader = DataLoader(EmotionDataset(train_split), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(EmotionDataset(val_split), batch_size=BATCH_SIZE)
test_loader = DataLoader(EmotionDataset(test_df, is_test=True), batch_size=BATCH_SIZE)

## 4. Model: DistilBERT + Classification Head
`AutoModelForSequenceClassification` with `problem_type='multi_label_classification'` adds a linear head on top of DistilBERT's `[CLS]` representation and uses `BCEWithLogitsLoss` internally.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    PRETRAINED,
    num_labels=num_labels,
    problem_type='multi_label_classification',
).to(DEVICE)

no_decay = ('bias', 'LayerNorm.weight')
grouped = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]
optimizer = torch.optim.AdamW(grouped, lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps,
)

## 5. Training Loop
Tracks val ROC-AUC each epoch. The full 3-epoch run is on Modal (`python -m modal run modal_train_bert.py`); this loop lets you sanity-check the architecture locally.

In [ ]:
def evaluate(loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
            ).logits
            preds.append(torch.sigmoid(logits).cpu().numpy())
            targets.append(batch['labels'].numpy())
    return roc_auc_score(np.vstack(targets), np.vstack(preds), average='macro')

best_auc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            labels=batch['labels'].to(DEVICE),
        )
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    val_auc = evaluate(val_loader)
    print(f'Epoch {epoch} | Val ROC AUC: {val_auc:.4f}')

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), 'model_best.pth')
        print(f'  ↑ new best, saved to model_best.pth')

print(f'\nBest val ROC-AUC: {best_auc:.4f}')

## 6. Generate Submission (local run)
For the final submission we load the Modal-trained `model_best.pth`; this cell just verifies the local checkpoint runs end-to-end.

In [ ]:
model.load_state_dict(torch.load('model_best.pth', map_location=DEVICE))
model.eval()
probs = []
with torch.no_grad():
    for batch in test_loader:
        logits = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
        ).logits
        probs.append(torch.sigmoid(logits).cpu().numpy())

submission = pd.DataFrame(np.vstack(probs), columns=emotion_cols)
submission.insert(0, 'ID', test_df['ID'].values)
submission.to_csv('submission.csv', index=False)
print('Submission file created: submission.csv')